# Your First Queries

In the explainer, we saw why organisations store data in databases rather than spreadsheets: enforced structure, concurrent access, and scale. We looked at schemas, tables, and data types. Now we get to talk to a database directly.

We are working with a metro network database for a city transport authority. The authority needs basic information about stations, lines, and zones before the team begins deeper analysis. Our job is to write the queries that pull that information out.

If you have used spreadsheet filters and sorting before, you will recognise a lot of what SQL does. The difference is that we write the instruction rather than clicking through a menu.

By the end of this notebook we will be able to:

- Retrieve rows and columns from a table using `SELECT`
- Filter rows with `WHERE` and a range of comparison operators
- Sort results with `ORDER BY` and limit output with `LIMIT`
- Combine conditions using `AND` and `OR`

---

## Setup

Run the cell below to install the SQL extension and connect to the database. We only need to do this once per session.

In [ ]:
%pip install -q jupysql

In [ ]:
%load_ext sql

In [ ]:
%sql sqlite:///../data/metro_transit.sqlite

---

## 1. SELECT: choosing columns

Every SQL query starts with `SELECT`. It tells the database which columns we want to see — similar to choosing which columns to display in a spreadsheet, except we write it out.

```sql
SELECT column1, column2 FROM table_name;
```

To get every column, use `SELECT *`.

The transport authority's database has a `stations` table. Start by pulling out just the name, line, and zone for each station.

In [ ]:
%%sql
SELECT name, line, zone FROM stations LIMIT 10;

That returned the first 10 rows with only the three columns we asked for. Now try `SELECT *` to see every column in the table:

In [ ]:
%%sql
SELECT * FROM stations LIMIT 5;

---

## 2. WHERE: filtering rows

We can see the full table, but the transport authority rarely needs everything at once. More often, someone asks a specific question: "Which stations are on the Red line?" or "What opened before 1980?" The `WHERE` clause filters rows to just the ones that match a condition.

In a spreadsheet, we would apply a column filter. In SQL, we write the condition directly.

### Equality and comparison

| Operator | Meaning |
|----------|---------|
| `=` | Equal to |
| `!=` or `<>` | Not equal to |
| `<` | Less than |
| `>` | Greater than |
| `<=` | Less than or equal |
| `>=` | Greater than or equal |

In [ ]:
%%sql
SELECT name, line, zone
FROM stations
WHERE line = 'Red';

In [ ]:
%%sql
SELECT name, opened_year
FROM stations
WHERE opened_year < 1980;

### LIKE: pattern matching

Sometimes we do not know the exact value we are looking for. Maybe the authority wants all stations with "Park" in the name. `LIKE` matches text patterns. Use `%` as a wildcard for any sequence of characters and `_` for a single character.

In [ ]:
%%sql
SELECT name, line
FROM stations
WHERE name LIKE '%Park%';

### BETWEEN: range filtering

The authority wants to know which stations opened in the 1990s. `BETWEEN` is a shorthand for `>=` and `<=` combined. Both endpoints are inclusive.

In [ ]:
%%sql
SELECT name, opened_year
FROM stations
WHERE opened_year BETWEEN 1990 AND 2000;

### IN: matching a list of values

What if we need stations on the Red line *or* the Blue line? We could write two conditions with `OR`, but `IN` is cleaner. It checks whether a value matches any item in a list.

In [ ]:
%%sql
SELECT name, line, zone
FROM stations
WHERE line IN ('Red', 'Blue');

### IS NULL and IS NOT NULL

Remember from the explainer that databases enforce data types and constraints. One consequence is that missing values have a specific representation: `NULL`. We cannot check for `NULL` with `=`. Instead, use `IS NULL` or `IS NOT NULL`.

The `staff` table has a `station_id` column that is `NULL` for control room staff who are not assigned to a specific station.

In [ ]:
%%sql
SELECT name, role
FROM staff
WHERE station_id IS NULL
LIMIT 10;

In [ ]:
%%sql
SELECT name, role, station_id
FROM staff
WHERE station_id IS NOT NULL
LIMIT 10;

---

## 3. Combining conditions: AND, OR

Real questions are rarely about a single condition. The authority might ask: "Which zone 1 stations are accessible?" That requires two conditions to be true at once. Use `AND` when both conditions must hold. Use `OR` when at least one must hold. Parentheses clarify the order of evaluation.

In [ ]:
%%sql
SELECT name, line, zone, has_accessibility
FROM stations
WHERE zone = 1 AND has_accessibility = 1;

In [ ]:
%%sql
SELECT name, line, zone
FROM stations
WHERE line = 'Green' OR zone = 4;

In [ ]:
%%sql
SELECT name, line, zone
FROM stations
WHERE (line = 'Red' OR line = 'Blue') AND zone <= 2;

Without the parentheses, SQL would evaluate `AND` before `OR`, producing different results. When mixing `AND` and `OR`, always use parentheses to make your intention explicit.

---

## 4. ORDER BY and LIMIT

So far, results come back in whatever order the database stores them. That is rarely useful for a report. `ORDER BY` sorts results by one or more columns. The default is ascending (`ASC`). Add `DESC` for descending.

`LIMIT` restricts how many rows are returned. Together, `ORDER BY` and `LIMIT` answer questions like "What are the five oldest stations on the network?"

In [ ]:
%%sql
SELECT name, opened_year
FROM stations
ORDER BY opened_year ASC
LIMIT 5;

Those are the five oldest stations on the network. Now the five newest:

In [ ]:
%%sql
SELECT name, opened_year
FROM stations
ORDER BY opened_year DESC
LIMIT 5;

We can sort by multiple columns. This sorts by zone first, then alphabetically by name within each zone — useful if the authority wants a station directory organised by zone:

In [ ]:
%%sql
SELECT name, line, zone
FROM stations
ORDER BY zone ASC, name ASC
LIMIT 15;

---

## 5. Putting it all together

A full query combines `SELECT`, `FROM`, `WHERE`, `ORDER BY`, and `LIMIT` in this order:

```sql
SELECT columns
FROM table
WHERE conditions
ORDER BY column ASC|DESC
LIMIT n;
```

Suppose the authority asks: "What are the 5 most recently opened accessible stations on the Red or Blue lines?" That is one question, but it uses everything we have learned so far:

In [ ]:
%%sql
SELECT name, line, zone, opened_year, has_accessibility
FROM stations
WHERE line IN ('Red', 'Blue')
  AND has_accessibility = 1
ORDER BY opened_year DESC
LIMIT 5;

---

## Exercises

Now it is your turn. Each exercise asks you to answer a question about the metro network by writing a SQL query. These are the kinds of questions the transport authority's analysts handle daily.

### Exercise 1: Stations on the Green line

Select the name, zone, and opened_year of every station on the Green line. Order results by zone ascending.

In [ ]:
%%sql
-- Your query here


### Exercise 2: Stations opened before 1975

Find all stations that opened before 1975. Return the name, line, and opened_year, sorted by opened_year ascending.

In [ ]:
%%sql
-- Your query here


### Exercise 3: Accessible stations in zone 1

Find all stations in zone 1 that have accessibility (`has_accessibility = 1`). Return the name and line.

In [ ]:
%%sql
-- Your query here


### Exercise 4: Stations with "Hill" in the name

Find all stations whose name contains the word "Hill". Return the name, line, and zone.

In [ ]:
%%sql
-- Your query here


---

## Summary

We have written our first SQL queries against a real database. We can now:

- Retrieve columns with `SELECT` and `SELECT *`
- Filter rows using `WHERE` with `=`, `<`, `>`, `!=`, `LIKE`, `BETWEEN`, `IN`, `IS NULL`, and `IS NOT NULL`
- Combine conditions with `AND` and `OR`, using parentheses to control evaluation order
- Sort results with `ORDER BY` (ascending and descending) and cap output with `LIMIT`

These are the building blocks of every SQL query we will write. In the next notebook, we will use **aggregation functions** to move beyond individual rows and start summarising ridership data across the network.